# Обучение модели ключевых точек (DXA)

Ноутбук рассчитан на Colab или локальную машину с GPU. Порядок: код → данные → конфиг → обучение → метрики → картинки.

**Данные в репозиторий не кладём.** Снимки медицинские: в Colab они попадают на серверы Google, поэтому
перед загрузкой убедитесь, что это допустимо регламентом конкурса. Если нет — обучайте локально.

Спецификация решения: `TODO.md`, устройство датасета: `DATASET.md`.

## 1. Код: клонирование репозитория

Токен нигде не печатается и не сохраняется: он не попадает ни в текст ошибки, ни в `.git/config` клона.
Если ячейка всё же покажет токен — считайте его скомпрометированным и отзовите.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = "github.com/theorcos239/LCT_2026.git"   # без https://
BRANCH = "points_detection"
WORKDIR = Path("/content") if Path("/content").exists() else Path.cwd()

def sh(*cmd, secret: str = "", **kw):
    """Запуск команды; secret вырезается из сообщений об ошибке."""
    hide = lambda s: s.replace(secret, "***") if secret else s
    print("$", hide(" ".join(map(str, cmd))))
    r = subprocess.run(list(map(str, cmd)), capture_output=True, text=True, **kw)
    if r.stdout:
        print(hide(r.stdout))
    if r.returncode:
        raise RuntimeError(hide(r.stderr.strip()) or f"код {r.returncode}")

def find_repo(start: Path) -> Path | None:
    for p in [start, *start.parents]:
        if (p / "src" / "dxa_qc").exists():
            return p
    return None

root = find_repo(Path.cwd())
if root is None:
    from getpass import getpass
    token = getpass("GitHub token (Enter — публичный репозиторий): ").strip()
    url = f"https://{'x-access-token:' + token + '@' if token else ''}{REPO}"
    target = WORKDIR / "dxa"
    try:
        sh("git", "clone", "--branch", BRANCH, "--depth", "1", url, target, secret=token)
    except RuntimeError as e:
        raise SystemExit(
            f"Клонирование не удалось: {e}\n"
            "Проверьте: существует ли ветка BRANCH (у нас master и points_detection, "
            "ветки main нет), есть ли у токена доступ к репозиторию и не истёк ли он."
        )
    # Токен не должен остаться в .git/config
    sh("git", "-C", target, "remote", "set-url", "origin", f"https://{REPO}")
    root = target

os.chdir(root)
sys.path.insert(0, str(root / "src"))
print("репозиторий:", root, "| ветка:",
      subprocess.run(["git", "branch", "--show-current"], capture_output=True, text=True).stdout.strip())

In [ ]:
# Зависимости. В Colab torch уже стоит, остальное быстро.
try:
    import pydicom, yaml, sklearn, scipy   # noqa: F401
except ImportError:
    sh(sys.executable, "-m", "pip", "install", "-q", "pydicom", "pyyaml",
       "scikit-learn", "scipy", "pillow", "openpyxl")

import torch
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "нет")

## 2. Данные

Нужен `data/keypoints.npz`. Варианты: он уже собран и лежит рядом, его кладут с Google Drive,
либо он собирается из сырых данных (`Исследования/` + HTML-сборка + JSON разметки).

In [ ]:
POINTS = "full"                      # full (29 каналов) | core (21) | minimal (14)

PACK = Path(f"data/keypoints_{POINTS}.npz")
DATA_DIR = None                      # папка на Диске: и данные, и результаты обучения

if Path("/content").exists():
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/dxa")    # поправьте путь под себя
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    # на Диске файл может лежать и под старым именем keypoints.npz (набор full)
    for src in (DATA_DIR / PACK.name, DATA_DIR / "keypoints.npz"):
        if not PACK.exists() and src.exists() and (src.name == PACK.name or POINTS == "full"):
            PACK.parent.mkdir(exist_ok=True)
            PACK.write_bytes(src.read_bytes())

if not PACK.exists():
    # Сборка из сырых данных: пути поправьте под своё размещение.
    sh(sys.executable, "-m", "dxa_qc.pack",
       "--annotations", *sorted(map(str, Path("Processing").glob("razmetka_*.json"))),
       "--html", "Processing/razmetka_all.html",
       "--studies", "НД_для_обучения/Исследования",
       "--points", POINTS, "--out", str(PACK),
       env={**os.environ, "PYTHONPATH": "src"})

from dxa_qc import pack as packmod
data = packmod.load(PACK)
print(f"{len(data['images'])} снимков, каналов {len(data['names'])} (набор {POINTS})")
import collections
print(collections.Counter(data["regions"]))
print("папка на Диске:", DATA_DIR)

## 3. Конфиг

Базовый конфиг лежит в `configs/keypoints.yaml`. Здесь его можно менять точечно — как из командной строки,
`--set train.epochs=300`. Копия итогового конфига сохранится рядом с весами.

In [ ]:
from dxa_qc import config as cfgmod

OVERRIDES = [
    f"data.points={POINTS}",
    f"data.pack={PACK}",
    "train.epochs=300",
    "train.batch_size=16",
    "train.folds=[0,1,2,3,4]",
    # out_dir задаётся ячейкой ниже, чтобы наборы точек не смешивались
    # POINTS сверху выбирает набор точек: full / core / minimal
    # "data.use_clahe=true",      # аблация
    # "train.amp=true",           # не для Pascal
]

cfg = cfgmod.load("configs/keypoints.yaml", OVERRIDES)
print(yaml_dump := __import__("yaml").safe_dump(cfg, allow_unicode=True, sort_keys=False))

## 4. Проверка перед долгим обучением

Тест на переобучение: модель учится на нескольких снимках без аугментаций и должна дойти
до единиц миллиметров. Это проверка кода, а не данных, и занимает пару минут.
Провал означает баг в пайплайне — запускать обучение на 300 эпох бессмысленно.

Полезно ещё и тем, что печатает, загрузились ли веса ImageNet.

In [ ]:
# Тест на переобучение: учим несколько снимков без аугментаций.
# Исправный код доходит до единиц миллиметров. Если ошибка стоит на месте —
# дело не в объёме данных, а в баге (так нашлась гауссиана без единицы в пике).
import numpy as np, torch
from torch.utils.data import DataLoader
from dxa_qc.dataset import KeypointDataset
from dxa_qc.heatmaps import decode
from dxa_qc.loss import focal_heatmap_loss
from dxa_qc.model import KeypointNet
from dxa_qc.transforms import AugmentConfig

N_IMAGES, EPOCHS, REGION, TARGET_MM = 4, 60, "hip", 2.0   # REGION=None — обе области

torch.manual_seed(cfg["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
idx = [i for i, r in enumerate(data["regions"]) if REGION in (None, str(r))][:N_IMAGES]
ds = KeypointDataset(data, idx, canvas=cfg["data"]["canvas"], sigma=cfg["heatmap"]["sigma"],
                     soft_mask_weight=cfg["loss"]["soft_mask_weight"],
                     aug=AugmentConfig(enabled=False), seed=cfg["seed"])
dl = DataLoader(ds, batch_size=len(idx))

model = KeypointNet(len(data["names"]), tuple(cfg["model"]["decoder_channels"]),
                    cfg["model"]["pretrained"]).to(device)
print("веса ImageNet загружены:", model.pretrained_loaded)
opt = torch.optim.AdamW(model.param_groups(cfg["train"]["lr_encoder"] * 10,
                                           cfg["train"]["lr_decoder"] * 10),
                        weight_decay=cfg["train"]["weight_decay"])

history = []
for epoch in range(EPOCHS):
    model.train()
    for b in dl:
        loss = focal_heatmap_loss(model(b["image"].to(device)), b["target"].to(device),
                                  b["weight"].to(device), cfg["loss"]["alpha"], cfg["loss"]["beta"])
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
        b = next(iter(dl))
        maps = torch.sigmoid(model(b["image"].to(device)))
        coords, conf = decode(maps, cfg["heatmap"]["decode_window"])
        vis, lab = b["visible"].numpy(), b["labeled"].numpy()
        err = np.linalg.norm(coords.cpu().numpy() - b["coords"].numpy(), axis=-1) * cfg["data"]["mm_per_px"]
        median, c = float(np.median(err[vis])), conf.cpu().numpy()
    history.append(median)
    if epoch % 10 == 9 or epoch == 0:
        print(f"эпоха {epoch+1:3d}  loss {loss.item():.4f}  медиана {median:6.2f} мм"
              f"  уверенность: свои {c[vis].mean():.2f} чужие {c[~lab].mean():.2f}")

best = min(history)
still_falling = len(history) > 10 and history[-1] < 0.9 * history[-11]
print(f"\nлучшая медиана {best:.2f} мм при пороге {TARGET_MM} мм, снимков {len(idx)}")
if best <= TARGET_MM:
    print("ПРОЙДЕН: код выучивает точки, можно запускать полное обучение.")
elif still_falling:
    print("НЕ СОШЛОСЬ: ошибка всё ещё падает — увеличьте EPOCHS и повторите.")
else:
    print("ПРОВАЛЕН: ошибка вышла на плато выше порога. Ищите баг в пайплайне, "
          "а не увеличивайте выборку.")

## 5. Куда писать результаты

Обучение идёт в локальную папку (запись быстрая), а после каждого фолда результаты копируются
на Диск — в ту же папку, где лежат данные. При обрыве Colab достаточно перезапустить ноутбук:
эта же ячейка вернёт чекпоинты с Диска, а обучение продолжится с последней сохранённой эпохи.

In [ ]:
import shutil

RUN_NAME = f"keypoints_{POINTS}"      # у каждого набора точек свой прогон
RUN_DIR = Path("runs") / RUN_NAME                       # локально: быстрые записи
DRIVE_RUN = (DATA_DIR / "runs" / RUN_NAME) if DATA_DIR else None
cfg["train"]["out_dir"] = str(RUN_DIR)

def sync(src: Path, dst: Path | None) -> None:
    if dst is None or not src.exists():
        return
    dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)

# Возврат прерванного прогона с Диска
if DRIVE_RUN and DRIVE_RUN.exists():
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_RUN, RUN_DIR, dirs_exist_ok=True)
    print("восстановлено с Диска:", sorted(p.name for p in RUN_DIR.iterdir()))
else:
    print("прежних чекпоинтов нет, начинаем с нуля")

## 6. Обучение

In [ ]:
from dxa_qc import train as T

def on_checkpoint(fold, epoch, out_dir):
    """Вызывается из обучения раз в checkpoint_every эпох — сразу кладём чекпоинты на Диск."""
    sync(Path(out_dir), DRIVE_RUN)
    print(f"  фолд {fold}, эпоха {epoch}: чекпоинт на Диске")

all_folds = list(cfg["train"]["folds"])
for fold in all_folds:
    cfg["train"]["folds"] = [fold]
    T.train_all(cfg, on_checkpoint=on_checkpoint)      # готовый фолд пропускается сам
    sync(RUN_DIR, DRIVE_RUN)
    print(f"фолд {fold} завершён и сохранён на Диск")

cfg["train"]["folds"] = all_folds
result = T.train_all(cfg)                              # все фолды готовы: только метрики
sync(RUN_DIR, DRIVE_RUN)
print("устройство:", result["device"], "| результаты:", result["out_dir"], "→", DRIVE_RUN)

## 7. Метрики

Главная таблица — строка на точку: среднее по каналам прячет провалы отдельных ориентиров.
Требования из `TODO.md`: точки оси не хуже 1 мм, `trochanter_minor` и `shaft_medial` около 0.5 мм,
отступы ROI — 2 мм достаточно.

In [ ]:
import pandas as pd
pd.set_option("display.width", 220, "display.max_rows", 60)

table = result["per_point"]
target = {**{f"l{i}_center": 1.0 for i in range(1, 5)},
          "trochanter_minor": 0.5, "shaft_medial": 0.5}
view = table.copy()
view["target_mm"] = view.point.map(target)
view["ok"] = view.median_mm <= view.target_mm.fillna(3.0)
view.round(2)

In [ ]:
print("Сквозные метрики (геометрия по предсказанным точкам против разметки):")
display(result["end_to_end"].round(2))
print("\nКалиброванные пороги видимости:")
display(result["thresholds"].to_frame().T.round(2))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

oof = result["oof"]
err = np.linalg.norm(oof["pred"] - oof["true"], axis=-1) * cfg["data"]["mm_per_px"]
vis = oof["visible"]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(err[vis], bins=40)
for t, c in [(1, "tab:green"), (3, "tab:orange"), (5, "tab:red")]:
    ax[0].axvline(t, color=c, ls="--", label=f"{t} мм")
ax[0].set(title="Ошибка точки, мм", xlabel="мм"); ax[0].legend()

order = table.sort_values("median_mm")
ax[1].barh(order.point, order.median_mm)
ax[1].set(title="Медиана по точкам, мм"); ax[1].invert_yaxis()
ax[1].tick_params(labelsize=7)
plt.tight_layout(); plt.show()

## 8. Картинки: предсказание против разметки

Зелёный — разметка, красный — предсказание, линия между ними показывает ошибку.
Пустой кружок — точка, которую модель считает невидимой.

In [ ]:
from dxa_qc.pack import LABELED

names = data["names"]

def draw(k, ax):
    j = oof["index"][k]
    ax.imshow(data["images"][j], cmap="gray", vmin=0, vmax=255)
    thr = result["thresholds"].to_numpy()
    for c, name in enumerate(names):
        if data["state"][j][c] != LABELED and oof["conf"][k][c] < thr[c]:
            continue
        found = oof["conf"][k][c] >= thr[c]
        if data["state"][j][c] == LABELED:
            ty, tx = oof["true"][k][c][1], oof["true"][k][c][0]
            ax.plot(tx, ty, "o", ms=5, mfc="none", mec="lime", mew=1.2)
            if found:
                ax.plot([tx, oof["pred"][k][c][0]], [ty, oof["pred"][k][c][1]], "-", color="yellow", lw=0.8)
        ax.plot(*oof["pred"][k][c], "o", ms=4, mfc="red" if found else "none", mec="red", mew=1.2)
    h, w = data["images"][j].shape
    ax.set_xlim(0, w); ax.set_ylim(h, 0)      # не растягиваем оси под точки за краем кадра
    e = err[k][vis[k]]
    ax.set_title(f"{data['regions'][j]} {data['ids'][j][:8]}\nмедиана {np.median(e):.1f} мм "
                 f"| точек {int(vis[k].sum())}", fontsize=8)
    ax.axis("off")

def sheet(indices, title):
    fig, axs = plt.subplots(1, len(indices), figsize=(4 * len(indices), 5), squeeze=False)
    for ax, k in zip(axs[0], indices):
        draw(k, ax)
    fig.suptitle(title, fontsize=11)
    plt.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

per_image = np.array([np.median(err[k][vis[k]]) if vis[k].any() else np.nan
                      for k in range(len(err))])
order = np.argsort(np.nan_to_num(per_image, nan=1e9))
sheet(order[:3].tolist(), "Лучшие")
sheet(order[max(0, len(order) - 3):].tolist(), "Худшие — смотреть в первую очередь")

In [ ]:
# Отдельно: как модель справляется с видимостью — выдуманные и пропущенные точки.
thr = result["thresholds"].to_numpy()
found = oof["conf"] >= thr
lab = oof["labeled"]
rows = []
for c, name in enumerate(names):
    rows.append(dict(point=name,
                     выдумано=int((found[:, c] & ~vis[:, c] & lab[:, c]).sum()),
                     пропущено=int((~found[:, c] & vis[:, c] & lab[:, c]).sum()),
                     размечено=int(lab[:, c].sum())))
pd.DataFrame(rows).query("выдумано > 0 or пропущено > 0")

## 9. Инференс по фолдам

Отдельная проверка готовой модели: веса подтягиваются с Диска, каждый фолд считается **на своей
валидации** его же весами — это честная оценка, без утечки. В отличие от метрик выше, здесь работает
полный инференс: TTA, калиброванные пороги видимости и определение области по «молчанию» каналов.

In [ ]:
import shutil
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from dxa_qc import metrics as M, pack as packmod
from dxa_qc.folds import load as load_folds
from dxa_qc.infer import Predictor
from dxa_qc.pack import ABSENT, LABELED

FOLDS = None          # None — все, у которых есть веса; либо список, например [0, 2]
N_SHOW = 3            # сколько картинок показывать на фолд

RUN_DIR = globals().get("RUN_DIR", Path("runs/keypoints"))
DRIVE_RUN = globals().get("DRIVE_RUN", None)
if DRIVE_RUN and Path(DRIVE_RUN).exists():                 # подтягиваем веса с Диска
    RUN_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copytree(DRIVE_RUN, RUN_DIR, dirs_exist_ok=True)
    print("скопировано с Диска:", DRIVE_RUN)

ckpts = {int(p.stem[4:]): p for p in sorted(RUN_DIR.glob("fold*.pt"))
         if p.stem[4:].isdigit()}
print("веса фолдов:", sorted(ckpts) or "не найдены")

data = globals().get("data") or packmod.load("data/keypoints.npz")
fold_of = dict(zip(*load_folds("folds.csv")[["study", "fold"]].values.T))
mm = 0.6

def run_fold_inference(fold):
    predictor = Predictor([str(ckpts[fold])])
    idx = [i for i, s in enumerate(data["studies"]) if fold_of.get(str(s), -1) == fold]
    pred, conf, regions = [], [], []
    for i in idx:
        out = predictor.predict(data["images"][i])
        pred.append(out["coords"]); conf.append(out["confidence"]); regions.append(out["region"])
    pred, conf = np.array(pred), np.array(conf)
    state = data["state"][idx]
    visible, labeled = state == LABELED, (state == LABELED) | (state == ABSENT)
    true = data["coords"][idx]
    table = M.per_point(data["names"], pred, conf, true, visible, labeled,
                        predictor.thresholds, mm)
    e2e = M.end_to_end(data["names"], pred, true, visible, np.array(data["regions"])[idx])
    region_ok = np.mean([r == data["regions"][i] for r, i in zip(regions, idx)])
    err = np.linalg.norm(pred - true, axis=-1) * mm
    return dict(fold=fold, idx=idx, pred=pred, conf=conf, true=true, visible=visible,
                err=err, table=table, e2e=e2e, region_ok=region_ok,
                thresholds=predictor.thresholds)

def show(res):
    err, vis = res["err"], res["visible"]
    per_image = np.array([np.median(err[k][vis[k]]) if vis[k].any() else np.nan
                          for k in range(len(res["idx"]))])
    order = np.argsort(np.nan_to_num(per_image, nan=1e9))
    picks = list(order[:N_SHOW]) + list(order[-N_SHOW:])
    fig, axs = plt.subplots(1, len(picks), figsize=(3.6 * len(picks), 4.6), squeeze=False)
    for ax, k in zip(axs[0], picks):
        i = res["idx"][k]
        img = data["images"][i]
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        for c in range(len(data["names"])):
            found = res["conf"][k][c] >= res["thresholds"][c]
            if vis[k][c]:
                tx, ty = res["true"][k][c]
                ax.plot(tx, ty, "o", ms=5, mfc="none", mec="lime", mew=1.2)
                if found:
                    ax.plot([tx, res["pred"][k][c][0]], [ty, res["pred"][k][c][1]],
                            "-", color="yellow", lw=0.8)
            if found:
                ax.plot(*res["pred"][k][c], "o", ms=4, mfc="red", mec="red")
        h, w = img.shape
        ax.set_xlim(0, w); ax.set_ylim(h, 0); ax.axis("off")
        ax.set_title(f"{data['regions'][i]} {data['ids'][i][:8]}\nмедиана {per_image[k]:.1f} мм", fontsize=8)
    fig.suptitle(f"фолд {res['fold']}: слева лучшие, справа худшие", fontsize=11)
    plt.tight_layout(rect=(0, 0, 1, 0.94)); plt.show()

results = []
for fold in (sorted(ckpts) if FOLDS is None else FOLDS):
    res = run_fold_inference(fold)
    results.append(res)
    good = res["err"][res["visible"]]
    print(f"\n=== фолд {fold}: снимков {len(res['idx'])}, медиана {np.median(good):.2f} мм, "
          f"p90 {np.percentile(good, 90):.2f} мм, область определена верно "
          f"{100 * res['region_ok']:.0f}%")
    display(res["table"].round(2))
    display(res["e2e"].round(2))
    show(res)

if results:
    summary = pd.concat([r["table"].assign(fold=r["fold"]) for r in results])
    display(summary.groupby("point")[["median_mm", "p90_mm", "pck@1mm", "pck@3mm", "f1_visible"]]
            .mean().round(2).sort_values("median_mm"))

## 10. Диагностика: сдвиг нумерации позвонков

Если медиана по точкам позвоночника около 1 мм, а p90 — 25–35 мм, это не «шумные снимки»:
шаг «позвонок + диск» в наших данных как раз 27–36 мм. Скорее всего, на части снимков модель
уверенно ставит всю цепочку **на один позвонок выше или ниже**.

Ячейка проверяет это прямо: для каждого снимка позвоночника сравнивает предсказание с разметкой,
сдвинутой на ±1 позвонок, и смотрит, какой вариант ближе.

In [ ]:
import numpy as np, pandas as pd

CHAIN = ["th12_bottom", "l1_center", "l2_center", "l3_center", "l4_center"]   # сверху вниз

names = list(data["names"])
ch = [names.index(n) for n in CHAIN]

rows = []
for res in results:                      # results — из ячейки инференса по фолдам
    for k, i in enumerate(res["idx"]):
        if data["regions"][i] != "spine" or not res["visible"][k][ch].all():
            continue
        true, pred = res["true"][k][ch], res["pred"][k][ch]
        step = np.median(np.linalg.norm(np.diff(true, axis=0), axis=1))      # шаг позвонка, px
        # сравниваем предсказание с разметкой как есть и со сдвинутой на один позвонок
        err = {0: np.linalg.norm(pred - true, axis=1).mean(),
               +1: np.linalg.norm(pred[1:] - true[:-1], axis=1).mean(),      # модель ниже на позвонок
               -1: np.linalg.norm(pred[:-1] - true[1:], axis=1).mean()}      # выше на позвонок
        best = min(err, key=err.get)
        rows.append(dict(fold=res["fold"], id=data["ids"][i], vert_shift=best,
                         err_mm=err[best] * 0.6, err_no_shift_mm=err[0] * 0.6,
                         step_mm=step * 0.6, flags=data["flags"][i]))

df = pd.DataFrame(rows)
if df.empty:
    print("нет снимков позвоночника с полной цепочкой точек")
else:
    print(f"снимков позвоночника: {len(df)}")
    print(df.vert_shift.value_counts().rename({0: "без сдвига", 1: "сдвиг вниз", -1: "сдвиг вверх"}).to_string())
    shifted = df[df.vert_shift != 0]
    print(f"\nдоля со сдвигом: {100 * len(shifted) / len(df):.0f}%")
    if len(shifted):
        print("на этих снимках ошибка падает со", f"{shifted.err_no_shift_mm.median():.1f} мм",
              "до", f"{shifted.err_mm.median():.1f} мм, если учесть сдвиг —",
              "значит модель нашла позвонки правильно, но пронумеровала иначе")
        display(shifted.sort_values("err_no_shift_mm", ascending=False).round(2).head(12))
    display(df.groupby("vert_shift")[["err_no_shift_mm", "err_mm", "step_mm"]].median().round(2))